# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k24csaiml1e2411265-wq/ML_starter_FlyrankAI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I reviewed the FlyRank research report **The State of AI-Driven SEO — FlyRank Data Report, April 2026**.

### Finding 1 — Broader query coverage is associated with stronger observed impressions

The report observes that pages with a moderate target-keyword match share can have higher average impressions than pages whose traffic is tightly concentrated on one target keyword. It recommends broad topical relevance rather than chasing 100% keyword match.

**Methodology question:** The finding is descriptive, so what exactly defines the outcome being compared — average impressions over the stated recent window, and are pages grouped using features measured in the same window? If a future outcome is being predicted, the label would need to come from a later, clearly separated window. I would also ask whether differences remain after accounting for content age, site/client mix, and other confounders.

**Validation question:** Because this is an observational comparison, do the grouping and validation design support an association claim only, or a predictive/generalization claim? The report appropriately treats the main findings as patterns rather than proof of causation. A grouped-by-site/client or time-aware validation would be especially useful if the finding is used as a predictive rule.

### Finding 2 — Freshness / refreshed-vs-stale pages show different observed performance

The report includes freshness and refreshed-vs-stale comparisons and reports statistically significant differences in observed impressions.

**Methodology question:** How is the freshness or stale/refreshed status defined, and is that status known before the performance window being measured? If the status is derived from the same period as impressions, the comparison can be descriptive but should not be treated as a forward-looking prediction.

**Validation question:** Does the validation design separate the period used to define freshness from the period used to measure the outcome? If not, the evidence supports an observed association, not necessarily a claim that refreshing content causes the later improvement. A time-aware before/after design or controlled experiment would strengthen a causal interpretation.

These are constructive methodology questions: they test whether the evidence supports the wording of the claim rather than grading the report.

In [4]:
import pandas as pd

# This cell records the audit questions as executable evidence.
paper_audit = pd.DataFrame([
    {
        'finding': 'Broader query coverage / moderate keyword-match share',
        'label_source': 'Observed impressions in a defined reporting window',
        'key_question': 'Is the outcome window separated from any feature window?',
        'claim_level': 'Association / pattern unless stronger design is shown'
    },
    {
        'finding': 'Freshness / refreshed-vs-stale performance difference',
        'label_source': 'Observed impressions in a defined reporting window',
        'key_question': 'Is freshness defined before the outcome window?',
        'claim_level': 'Association unless time-aware or experimental evidence supports causality'
    }
])

display(paper_audit)
assert len(paper_audit) == 2

,finding,label_source,key_question,claim_level
0,Broader query coverage / moderate keyword-matc...,Observed impressions in a defined reporting wi...,Is the outcome window separated from any featu...,Association / pattern unless stronger design i...
1,Freshness / refreshed-vs-stale performance dif...,Observed impressions in a defined reporting wi...,Is freshness defined before the outcome window?,Association unless time-aware or experimental ...


## 2. My model under an honest split (before/after)

**Before:** a random row split, similar to a standard train/test split, can place rows from the same client in both sets. This may make the result look stronger because client-specific patterns can be shared across train and test.

**After:** a client-grouped split keeps every client entirely in train or test. This directly tests whether the model transfers to unseen clients.

The model is the Week-5 Random Forest for the same Content Refresh lane. The feature window is February 2026 and the outcome window is March 2026, so the prediction inputs precede the measured outcome.

In [3]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn
import os,getpass,duckdb,pandas as pd,numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score,roc_auc_score,classification_report

HF_TOKEN=os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN=userdata.get('HF_TOKEN')
    except Exception: pass
HF_TOKEN=HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
assert HF_TOKEN
con=duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL='hf://datasets/FlyRank/internship-warehouse'
DAILY=f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
QUERY=f"read_parquet('{REL}/fact_content_query_90d.parquet')"
print('Connected to FlyRank warehouse.')

# February features -> March outcome. No March information is used as a feature.
data=con.sql(f"""
WITH base AS (
 SELECT client_hash_id,content_hash_id,
        SUM(gsc_impressions) AS imp_prev30,
        SUM(gsc_clicks) AS clk_prev30,
        AVG(gsc_avg_position) AS pos_prev30,
        STDDEV_SAMP(gsc_avg_position) AS pos_volatility
 FROM {DAILY}
 WHERE report_date>=DATE '2026-02-01' AND report_date<DATE '2026-03-01'
 GROUP BY 1,2 HAVING SUM(gsc_impressions)>=100
),
queries AS (
 SELECT content_hash_id,
        SUM(impressions_90d) AS kept_impressions,
        MAX(impressions_90d) AS top_query_impressions,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share
 FROM {QUERY} GROUP BY content_hash_id
),
outcome AS (
 SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) AS imp_future30
 FROM {DAILY}
 WHERE report_date>=DATE '2026-03-01' AND report_date<DATE '2026-04-01'
 GROUP BY 1,2
)
SELECT b.*,q.visible_queries,q.rare_share,q.anon_share,
       CASE WHEN q.kept_impressions>0 THEN q.top_query_impressions*1.0/q.kept_impressions ELSE NULL END AS top_query_share,
       o.imp_future30
FROM base b LEFT JOIN queries q USING(content_hash_id)
INNER JOIN outcome o USING(client_hash_id,content_hash_id)
""").df()
data['ctr_prev30']=np.where(data.imp_prev30>0,data.clk_prev30/data.imp_prev30,np.nan)
data['is_declining']=(data.imp_future30<0.8*data.imp_prev30).astype(int)
feature_cols=['imp_prev30','clk_prev30','ctr_prev30','pos_prev30','pos_volatility','visible_queries','rare_share','anon_share','top_query_share']
model_data=data.dropna(subset=['client_hash_id','is_declining']).copy()
print(f'Rows: {len(model_data):,}; clients: {model_data.client_hash_id.nunique():,}; decline rate: {model_data.is_declining.mean():.3f}')

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 76,837; clients: 34; decline rate: 0.182


In [5]:
def fit_eval(train_df,test_df):
    pipe=Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('model',RandomForestClassifier(n_estimators=250,max_depth=8,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=-1))
    ])
    pipe.fit(train_df[feature_cols],train_df['is_declining'])
    pred=pipe.predict(test_df[feature_cols])
    prob=pipe.predict_proba(test_df[feature_cols])[:,1]
    return pipe,pred,prob

def metrics(y,pred,prob):
    return {'F1':f1_score(y,pred,zero_division=0),'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'accuracy':accuracy_score(y,pred),'ROC-AUC':roc_auc_score(y,prob)}

# BEFORE: random row split.
tr,te=train_test_split(model_data,test_size=0.25,random_state=42,stratify=model_data['is_declining'])
random_model,random_pred,random_prob=fit_eval(tr,te)
random_metrics=metrics(te.is_declining,random_pred,random_prob)

# AFTER: grouped by client.
gss=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
tri,tei=next(gss.split(model_data,model_data['is_declining'],groups=model_data['client_hash_id']))
gtr=model_data.iloc[tri]; gte=model_data.iloc[tei]
group_model,group_pred,group_prob=fit_eval(gtr,gte)
group_metrics=metrics(gte.is_declining,group_pred,group_prob)

comparison=pd.DataFrame([{'validation':'Random row split (before)',**random_metrics},{'validation':'Client-grouped split (after)',**group_metrics}])
display(comparison.round(3))
print('Client overlap in grouped split:',len(set(gtr.client_hash_id)&set(gte.client_hash_id)))
assert len(set(gtr.client_hash_id)&set(gte.client_hash_id))==0

,validation,F1,precision,recall,accuracy,ROC-AUC
0,Random row split (before),0.447,0.340,0.652,0.706,0.754
1,Client-grouped split (after),0.317,0.196,0.827,0.529,0.705


Client overlap in grouped split: 0


### Before/after interpretation

The random split is useful as a reference point, but it is not the strongest estimate of cross-client generalization. The grouped result is the more relevant validation number for a claim about transfer to unseen clients.

If the grouped score drops, that is not a failure of the project. It is evidence that the easier split was optimistic. The reported conclusion should use the grouped result and describe the model as directional decision-support.

In [6]:
print('Grouped-test classification report:')
print(classification_report(gte.is_declining,group_pred,digits=3,zero_division=0))
print('Train clients:',gtr.client_hash_id.nunique(),'Test clients:',gte.client_hash_id.nunique())
print('Grouped test rows:',len(gte))

Grouped-test classification report:
              precision    recall  f1-score   support

           0      0.948     0.484     0.641      7375
           1      0.196     0.827     0.317      1124

    accuracy                          0.529      8499
   macro avg      0.572     0.655     0.479      8499
weighted avg      0.849     0.529     0.598      8499

Train clients: 25 Test clients: 9
Grouped test rows: 8499


## 3. Leakage audit

The final feature set must be available at the decision moment. The March outcome is used only to construct the label for evaluation; it must never enter the feature matrix.

In [7]:
# Explicit feature audit.
feature_audit=pd.DataFrame([
 {'feature':c,'source_window':'February 2026 or historical query snapshot','allowed_before_decision':True,'reason':'Known before the March outcome window'} for c in feature_cols
])
feature_audit.loc[len(feature_audit)]={'feature':'imp_future30','source_window':'March 2026','allowed_before_decision':False,'reason':'Outcome window; used only to create the label'}
display(feature_audit)

assert 'imp_future30' not in feature_cols
assert not any(any(term in c.lower() for term in ['future','label','target','outcome']) for c in feature_cols)
print('Leakage audit: PASSED — future outcome is excluded from model features.')

,feature,source_window,allowed_before_decision,reason
0,imp_prev30,February 2026 or historical query snapshot,True,Known before the March outcome window
1,clk_prev30,February 2026 or historical query snapshot,True,Known before the March outcome window
2,ctr_prev30,February 2026 or historical query snapshot,True,Known before the March outcome window
3,pos_prev30,February 2026 or historical query snapshot,True,Known before the March outcome window
4,pos_volatility,February 2026 or historical query snapshot,True,Known before the March outcome window
5,visible_queries,February 2026 or historical query snapshot,True,Known before the March outcome window
6,rare_share,February 2026 or historical query snapshot,True,Known before the March outcome window
7,anon_share,February 2026 or historical query snapshot,True,Known before the March outcome window
8,top_query_share,February 2026 or historical query snapshot,True,Known before the March outcome window
9,imp_future30,March 2026,False,Outcome window; used only to create the label


Leakage audit: PASSED — future outcome is excluded from model features.


In [8]:
# Check that the model's actual feature matrix contains only the declared features.
actual_features=list(gtr[feature_cols].columns)
print('Model features:',actual_features)
assert actual_features==feature_cols
print('Final feature-set check: PASSED')

# Search for suspicious column names in the complete working frame.
suspicious=[c for c in model_data.columns if any(term in c.lower() for term in ['future','label','target','outcome'])]
print('Suspicious outcome-related columns in working data:',suspicious)
print('These columns are not included in feature_cols.')
assert 'imp_future30' not in feature_cols

Model features: ['imp_prev30', 'clk_prev30', 'ctr_prev30', 'pos_prev30', 'pos_volatility', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
Final feature-set check: PASSED
Suspicious outcome-related columns in working data: ['imp_future30']
These columns are not included in feature_cols.


## 4. Claim rewrite

### Earlier / too-strong version

> **The Random Forest predicts which pages will decline in search performance and can identify content that should be refreshed.**

### Safer evidence-based version

> **In this experiment, the Random Forest measured a directional ability to distinguish content items with a >20% March-impression decline from those without that measured decline. Performance was evaluated on client-held-out data, so the result should be treated as decision-support rather than proof that the model will generalize to every client or that refreshing a page will cause recovery.**

### What changed

- **“Measured”** replaces an unconditional claim of predictive success.
- The exact outcome definition (**>20% impression decline**) is stated.
- **Client-held-out validation** is named.
- **Decision-support** replaces an operational guarantee.
- No causal claim is made about refreshing content.

This wording is intentionally conservative because validation quality and observational data limit what the experiment can establish.

In [9]:
claim_audit=pd.DataFrame([
 {'old_claim':'Predicts which pages will decline and identifies pages that should be refreshed','problem':'Too broad; does not state validation population or causal limitation','safe_claim':'Measured directional discrimination on a client-held-out test set for the defined >20% decline outcome.'},
 {'old_claim':'Refreshing a flagged page will improve performance','problem':'Causal claim not tested by this observational model','safe_claim':'The score can support a review queue; whether a refresh improves performance requires a separate outcome-aware evaluation or experiment.'}
])
display(claim_audit)

,old_claim,problem,safe_claim
0,Predicts which pages will decline and identifi...,Too broad; does not state validation populatio...,Measured directional discrimination on a clien...
1,Refreshing a flagged page will improve perform...,Causal claim not tested by this observational ...,The score can support a review queue; whether ...


## Self-check

Before you submit, confirm each line honestly:

- [x] Two paper findings are named with constructive methodology questions
- [x] Before/after validation is shown: random row split vs client-grouped split
- [x] The grouped split has zero client overlap
- [x] Future-window outcome is excluded from the feature set
- [x] Real held-out failure metrics are reported
- [x] Claims use observed, measured, directional, and decision-support language
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] Committed to my repo under `work/notebooks/w06_validation_audit.ipynb`